# Chapter 1 Advanced Lab: Build a Quantum Momentum Index

**Applied Quantum Computing: A Leader's Guide to the Next Computing Revolution**  
Dr. Ernesto Lee | [Back to Book](https://liquid-books.github.io/applied-quantum-computing)

---

## Objective
Stop reading forecasts. Start measuring the field with data.

You will pull three real data streams — academic paper counts, open-source SDK activity, and funding rounds — normalize them, combine them into a single **Quantum Momentum Index**, and overlay it against the Gartner Hype Cycle regions.

## Deliverable
A plot of the Quantum Momentum Index over time (2015–present) plus a **500-word memo** answering:  
> *Is the field accelerating, plateauing, or in retreat — and what would change your mind?*

## Estimated Time
60–120 minutes

---

In [ ]:
# Run this first — installs required packages (~30 seconds in Colab)
!pip install -q requests pandas matplotlib numpy

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

print('Libraries loaded successfully.')

## Part 1: arXiv Paper Counts (2015–Present)

We query the arXiv API for papers tagged `quant-ph` (quantum physics/computing) by year.

In [ ]:
def fetch_arxiv_counts(start_year=2015, end_year=None):
    """Fetch quarterly paper counts from arXiv for quantum computing topics."""
    if end_year is None:
        end_year = datetime.now().year
    
    counts = {}
    base_url = 'http://export.arxiv.org/api/query'
    
    for year in range(start_year, end_year + 1):
        params = {
            'search_query': 'cat:quant-ph AND ti:quantum+computing',
            'start': 0,
            'max_results': 0,  # We only need the total count
            'submittedDate': f'[{year}01010000+TO+{year}12312359]'
        }
        try:
            resp = requests.get(base_url, params=params, timeout=10)
            # Parse total results from feed
            import xml.etree.ElementTree as ET
            root = ET.fromstring(resp.text)
            ns = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}
            total = root.find('opensearch:totalResults', ns)
            counts[year] = int(total.text) if total is not None else 0
            print(f'  {year}: {counts[year]:,} papers')
        except Exception as e:
            print(f'  {year}: Error ({e}) — using estimate')
            counts[year] = None
    
    return counts

print('Fetching arXiv paper counts...')
arxiv_counts = fetch_arxiv_counts(2015)
arxiv_series = pd.Series(arxiv_counts).dropna()
print(f'\nTotal papers found across all years: {arxiv_series.sum():,.0f}')

## Part 2: GitHub Commit Activity — Qiskit, Cirq, PennyLane

We use the GitHub API to pull commit counts per year for the three largest quantum SDKs.  
No API key required for public repos (rate-limited to 60 req/hr unauthenticated).

In [ ]:
def fetch_github_commits(owner, repo, start_year=2015):
    """Fetch annual commit counts for a GitHub repo."""
    counts = {}
    headers = {'Accept': 'application/vnd.github.v3+json'}
    
    for year in range(start_year, datetime.now().year + 1):
        url = f'https://api.github.com/repos/{owner}/{repo}/commits'
        params = {
            'since': f'{year}-01-01T00:00:00Z',
            'until': f'{year}-12-31T23:59:59Z',
            'per_page': 1
        }
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=10)
            # GitHub returns Link header with last page = total pages
            link = resp.headers.get('Link', '')
            if 'last' in link:
                import re
                last_page = int(re.search(r'page=(\d+)>; rel="last"', link).group(1))
                counts[year] = last_page
            elif resp.json():
                counts[year] = len(resp.json())
            else:
                counts[year] = 0
        except Exception as e:
            counts[year] = 0
    
    return counts

repos = [
    ('Qiskit', 'Qiskit', 'qiskit'),
    ('Cirq', 'quantumlib', 'Cirq'),
    ('PennyLane', 'PennyLaneAI', 'pennylane'),
]

github_data = {}
for name, owner, repo in repos:
    print(f'Fetching {name} commits...')
    github_data[name] = fetch_github_commits(owner, repo)

github_df = pd.DataFrame(github_data)
github_combined = github_df.sum(axis=1)  # Total commits across all 3 SDKs
print('\nAnnual commits (combined):')
print(github_combined)

## Part 3: Funding Rounds Data

We load a provided CSV of disclosed quantum-company funding rounds by year.

In [ ]:
# Quantum funding data (USD millions, disclosed rounds)
# Sources: PitchBook, Crunchbase public data, press releases
funding_data = {
    2015: 105, 2016: 149, 2017: 241, 2018: 450,
    2019: 612, 2020: 787, 2021: 1420, 2022: 1850,
    2023: 1100, 2024: 1650, 2025: 2100
}
funding_series = pd.Series(funding_data)
print('Annual quantum funding (USD millions):')
print(funding_series)

## Part 4: Normalize and Combine → Quantum Momentum Index

We normalize each series to a 0–100 scale (min-max normalization) and average them into a single index.

In [ ]:
def normalize_series(s):
    """Min-max normalize a pandas Series to 0-100."""
    s = s.dropna()
    return (s - s.min()) / (s.max() - s.min()) * 100

# Align all series to same year index
years = sorted(set(arxiv_series.index) & set(github_combined.index) & set(funding_series.index))

arxiv_norm = normalize_series(arxiv_series.reindex(years))
github_norm = normalize_series(github_combined.reindex(years))
funding_norm = normalize_series(funding_series.reindex(years))

# TODO: Experiment with different weights. Does funding drive the index? Or papers?
weights = {'arxiv': 1/3, 'github': 1/3, 'funding': 1/3}

qmi = (arxiv_norm * weights['arxiv'] +
       github_norm * weights['github'] +
       funding_norm * weights['funding'])

print('Quantum Momentum Index by year:')
for year, val in qmi.items():
    print(f'  {year}: {val:.1f}')

## Part 5: Plot with Gartner Hype Cycle Regions

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Plot the three components
ax.plot(years, arxiv_norm.reindex(years), '--', color='steelblue', alpha=0.5, label='arXiv Papers (normalized)')
ax.plot(years, github_norm.reindex(years), '--', color='darkorange', alpha=0.5, label='SDK Commits (normalized)')
ax.plot(years, funding_norm.reindex(years), '--', color='gray', alpha=0.5, label='Funding Rounds (normalized)')

# Plot the combined index
ax.plot(years, qmi.reindex(years), 'o-', color='navy', linewidth=2.5,
        markersize=8, label='Quantum Momentum Index', zorder=5)

# Gartner Hype Cycle region overlays
# TODO: Adjust these year boundaries based on your analysis
hype_regions = [
    (2015, 2018, '#fff3cd', 'Innovation Trigger'),
    (2018, 2022, '#fde8e8', 'Peak of Inflated Expectations'),
    (2022, 2024, '#e8f0fe', 'Trough of Disillusionment'),
    (2024, max(years)+0.9, '#e8fde8', 'Slope of Enlightenment'),
]
for start, end, color, label in hype_regions:
    ax.axvspan(start, end, alpha=0.25, color=color, label=label)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Index Score (0–100)', fontsize=12)
ax.set_title('Quantum Momentum Index: 2015–Present', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(min(years) - 0.5, max(years) + 0.5)
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('quantum_momentum_index.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as quantum_momentum_index.png')

## Deliverable: Your 500-Word Memo

Write your memo in the cell below. Answer:
1. **Is the field accelerating, plateauing, or in retreat?** Cite your index values.
2. **Which component is driving the index most?** What does that tell you?
3. **What would change your mind?** Name two specific signals that would indicate a reversal.

---
**Target:** ~500 words. Write as a business memo, not a lab report.

**YOUR MEMO HERE:**

TO: [Instructor / Peer Reviewer]  
FROM: [Your Name]  
RE: Quantum Momentum Index Analysis  
DATE: [Today's Date]

*[Write your 500-word analysis here...]*